# Lost in Transcription — Spanish–Nahuatl

This is the dedicated Spanish–Nahuatl RunPod workflow. Do **not** use the Indonesian–Javanese notebook for this competition.

Run cells top-to-bottom. Select models by speaker-disjoint competition-dev WER; never by training loss or a random clip split.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
from getpass import getpass

ROOT = Path('/workspace/Lost_In_Transcription_Indonesian_Javanese_Track')
if not ROOT.is_dir():
    raise FileNotFoundError(f'Update ROOT for this pod: {ROOT}')
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT))

SPNH_DEV_DIR = Path('/workspace/nahuatl_dev')
TETEL_DIR = Path('/workspace/tetelancingo_nahuatl')
ARTIFACTS = ROOT / 'artifacts' / 'spnh'
ARTIFACTS.mkdir(parents=True, exist_ok=True)

# Preferred: set MDC_API_KEY as a RunPod secret/environment variable.
# Fallback: this asks privately and keeps the value only in this kernel.
if not os.environ.get('MDC_API_KEY'):
    os.environ['MDC_API_KEY'] = getpass('MDC API key (not saved): ')
if not os.environ['MDC_API_KEY'].strip():
    raise RuntimeError('MDC_API_KEY is required to download Tetelancingo')


## Download Tetelancingo

Skip this cell only if `TETEL_DIR` already contains the extracted corpus. The API key is used only on RunPod to obtain the public training corpus; it must never be copied into the final ZIP.

In [ ]:
if not TETEL_DIR.exists():
    subprocess.run(
        [sys.executable, str(ROOT / 'scripts' / 'download_mdc_dataset.py'), '--out', str(TETEL_DIR)],
        check=True,
        env=os.environ.copy(),
    )
else:
    print(f'Reusing {TETEL_DIR}')


## Build manifests and the held-out speaker split

The competition test speakers and conversations do not overlap development data. This makes speaker-disjoint WER the decision metric. The Tetelancingo `sentence` field is used as normalized Nahuatl supervision; Spanish translations are never training targets.

In [ ]:
from spanish_nahuatl import prepare
from lit_jv import prompts
from lit_jv.decode import DecodeConfig, FallbackConfig, Qwen3AsrDecoder, build_backend
from lit_jv.scoring import wer_report
from lit_jv.train_lora import AugmentSpec, LoraSpec, StageSpec, TrainSpec, merge_and_save, train

dev_rows = prepare.load_competition_dev(SPNH_DEV_DIR)
dev_train, dev_eval = prepare.holdout_speakers(dev_rows, count=1, seed=2026)
tetel_train = prepare.load_tetelancingo(TETEL_DIR, split='train')
tetel_eval = prepare.load_tetelancingo(TETEL_DIR, split='test')
fit_rows = tetel_train + dev_train
terms = prepare.glossary(fit_rows)
(ARTIFACTS / 'glossary.json').write_text(json.dumps(terms, ensure_ascii=False, indent=2), encoding='utf-8')
summary = {
    'dev_train': len(dev_train), 'dev_eval': len(dev_eval),
    'tetel_train': len(tetel_train), 'tetel_eval': len(tetel_eval),
    'glossary': len(terms),
    'orthography_flags': prepare.orthography_audit(tetel_train),
}
summary


## Zero-shot gate

Record this WER. A fine-tune that fails to beat it is rejected.

In [ ]:
def score_rows(model_path, rows, prompt, name, chunk_size=16):
    backend = build_backend(model_path, prefer='vllm', log=print)
    decoder = Qwen3AsrDecoder(backend, fallback=FallbackConfig(enabled=False), log=print)
    try:
        hypotheses = decoder.decode_pass(
            rows, DecodeConfig(prompt=prompt, name=name, temperature=0.0), chunk_size=chunk_size
        )
        report = wer_report([row['raw_text'] for row in rows], [hyp.text for hyp in hypotheses])
        print(report.as_row(name))
        return report
    finally:
        decoder.close()

baseline_prompt = prompts.PromptConfig(
    prose='codeswitch_spnh', style_tag=prompts.STYLE_VOICE_NOTE_SP_NH,
    glossary=tuple(terms), max_glossary_terms=128, force_language='Spanish',
)
baseline = score_rows('Qwen/Qwen3-ASR-1.7B', dev_eval, baseline_prompt, 'qwen-zero-shot')


## Train the selected recipe

Stage 1 learns Nahuatl vocabulary/acoustics from Tetelancingo. Stage 2 prioritizes the in-domain Spanish–Nahuatl voice-note style. Generation WER is checkpoint selection; teacher-forced evaluation stays disabled because raw audio rows are not pre-tokenized examples.

In [ ]:
spec = TrainSpec(
    model_path='Qwen/Qwen3-ASR-1.7B',
    output_dir=str(ARTIFACTS / 'lora'),
    target_language='Spanish',
    lora=LoraSpec(r=16, alpha=32, adapt_text_decoder=True, adapt_audio_encoder=False),
    augment=AugmentSpec(corpora=('tetelancingo',)),
    prompt=baseline_prompt,
    eval_steps=0, wer_eval_steps=10, wer_eval_clips=len(dev_eval), wer_eval_batch_size=4,
    run_name='spnh-qwen3asr',
)
stages = (
    StageSpec('s1-tetelancingo', {'tetelancingo': 1.0, 'spnh_dev': 1.25}, epochs=2.0, learning_rate=1.0e-4, batch_size=4, grad_accumulation=8),
    StageSpec('s2-voice-note', {'spnh_dev': 1.0, 'tetelancingo': 0.15}, epochs=5.0, learning_rate=3.0e-5, batch_size=4, grad_accumulation=8),
)
selected = train(fit_rows, spec, stages=stages, save_adapter_as='selected', eval_rows=dev_eval)
selected['best_wer']


## Select, final-fit, and package

If the assertion fails, stop: alter one variable and repeat from the zero-shot gate. If it passes, the final fit freezes the selected recipe and uses all released training data.

In [ ]:
merged = merge_and_save(spec, ARTIFACTS / 'lora' / 'selected', ARTIFACTS / 'merged')
selected_dev = score_rows(merged, dev_eval, spec.prompt, 'selected-dev')
selected_tetel = score_rows(merged, tetel_eval, spec.prompt, 'selected-tetel')
assert selected_dev.wer < baseline.wer, 'Reject this recipe: it did not beat zero-shot Qwen.'

final_rows = prepare.load_tetelancingo(TETEL_DIR, split=None) + dev_rows
final_spec = TrainSpec(**{**spec.__dict__, 'output_dir': str(ARTIFACTS / 'final-lora'), 'wer_eval_steps': 0})
train(final_rows, final_spec, stages=stages, save_adapter_as='final', eval_rows=None)
final_merged = merge_and_save(final_spec, ARTIFACTS / 'final-lora' / 'final', ARTIFACTS / 'final-merged')
subprocess.run([
    sys.executable, str(ROOT / 'spanish_nahuatl' / 'package_submission.py'),
    '--model', str(final_merged), '--glossary', str(ARTIFACTS / 'glossary.json'),
], check=True)
print(ARTIFACTS / 'submission.zip')
